# Importing the necessary libraries

In [8]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np
import optuna
import os

In [9]:
import torch
import torch.nn as nn
import torch.optim as optim

In [10]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

In [11]:
print(device)

mps


# Loading the datasets

In [12]:
blr_df = pd.read_csv('../Data/Processed/blr_df_enhanced.csv')
hyd_df = pd.read_csv('../Data/Processed/hyd_df_enhanced.csv')
pune_df = pd.read_csv('../Data/Processed/pune_df_enhanced.csv')

In [13]:
blr_df

,Date,DPT,AP,WS,WSD,AT,Albedo,evaporation_from_bare_soil_sum,evaporation_from_vegetation_transpiration_sum,NDVI,precipitation,surface_net_solar_radiation_sum,surface_thermal_radiation_downwards_sum,volumetric_soil_water_layer_1,LST
0,2003-01-01,12.498648,916.589625,0.608896,327.505071,294.583644,0.164460,-0.000866,-0.000029,0.265743,0.000000,1.427781e+07,31100000.0,0.192649,30.939528
1,2003-01-02,12.825784,918.004624,3.006925,290.310937,294.396121,0.164931,-0.000744,-0.000054,0.321507,0.006589,1.279035e+07,31500000.0,0.192116,31.179113
2,2003-01-03,12.970981,918.453619,2.872020,279.921010,293.651644,0.165501,-0.000771,-0.000051,0.402250,0.000000,1.492127e+07,30700000.0,0.191724,33.045705
3,2003-01-04,12.445614,917.966755,2.678422,274.952584,294.383796,0.165904,-0.000971,-0.000048,0.370532,0.000000,1.747538e+07,29400000.0,0.191393,38.266933
4,2003-01-05,14.071074,918.579933,3.104962,275.257170,294.637442,0.165906,-0.000843,-0.000050,0.407205,0.000000,1.584263e+07,30200000.0,0.190960,33.185394
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6553,2020-12-25,13.500707,916.151180,2.152634,261.457477,292.513759,0.140036,-0.002592,-0.000046,0.421393,0.000000,1.671576e+07,29200000.0,0.297770,32.118758
6554,2020-12-26,12.901577,917.194971,2.312996,253.686322,292.035179,0.140023,-0.002544,-0.000056,0.421393,0.000000,1.764899e+07,28000000.0,0.289527,32.068009
6555,2020-12-27,12.950693,916.302854,2.260313,252.095788,292.771807,0.139925,-0.002661,-0.000051,0.421393,0.000000,1.757099e+07,28200000.0,0.281540,34.539327
6556,2020-12-28,10.830827,916.237697,2.003774,254.229795,292.428852,0.139896,-0.002655,-0.000055,0.421393,0.000000,1.724660e+07,28200000.0,0.273806,31.310399


In [14]:
pune_df

,Date,DPT,AP,WS,WSD,AT,Albedo,evaporation_from_bare_soil_sum,evaporation_from_vegetation_transpiration_sum,NDVI,precipitation,surface_net_solar_radiation_sum,surface_thermal_radiation_downwards_sum,volumetric_soil_water_layer_1,LST
0,2003-01-01,7.100958,941.961323,1.933461,262.137755,291.684654,0.136310,-0.000517,-0.000160,0.457425,0.000000,1.617383e+07,26800000.0,0.148610,34.246204
1,2003-01-02,10.484796,942.675433,1.436056,300.935828,294.195680,0.136204,-0.000475,-0.000130,0.359140,0.000000,1.418103e+07,30800000.0,0.148434,35.734294
2,2003-01-03,12.606104,942.881457,1.486452,300.281597,295.828579,0.136274,-0.000444,-0.000083,0.333687,0.000000,1.168451e+07,32600000.0,0.148452,32.590960
3,2003-01-04,12.646417,943.009227,1.479805,280.080236,295.986278,0.136218,-0.000461,-0.000058,0.361981,0.000000,1.190128e+07,32100000.0,0.148349,35.435619
4,2003-01-05,13.066706,943.762527,2.159695,268.877282,296.071805,0.135858,-0.000515,-0.000077,0.361361,0.000000,1.382363e+07,31200000.0,0.148208,34.235347
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6553,2020-12-25,12.096190,941.053885,2.219879,278.843058,294.637357,0.127867,-0.001178,-0.000121,0.465578,0.125304,1.532697e+07,29600000.0,0.171530,33.209944
6554,2020-12-26,13.381471,942.263093,1.319014,277.169354,295.234928,0.127843,-0.001168,-0.000078,0.465578,0.000000,1.530680e+07,29900000.0,0.168792,31.352555
6555,2020-12-27,13.819016,941.287692,0.585305,157.827979,295.475096,0.127817,-0.001195,-0.000068,0.465578,0.000000,1.524516e+07,30000000.0,0.166471,33.862652
6556,2020-12-28,13.831078,940.630939,0.346885,190.861072,295.019423,0.127371,-0.001151,-0.000072,0.465578,0.000000,1.555207e+07,29500000.0,0.164420,32.318171


In [15]:
hyd_df

,Date,DPT,AP,WS,WSD,AT,Albedo,evaporation_from_bare_soil_sum,evaporation_from_vegetation_transpiration_sum,NDVI,precipitation,surface_net_solar_radiation_sum,surface_thermal_radiation_downwards_sum,volumetric_soil_water_layer_1,LST
0,2003-01-01,9.801921,952.321702,2.702138,228.989644,294.587935,0.150061,-0.000407,-0.000179,0.230621,0.0,1.548250e+07,28900000.0,0.119923,35.253027
1,2003-01-02,12.894156,953.896739,2.974241,287.669929,295.364687,0.151337,-0.000302,-0.000127,0.197456,0.0,1.177422e+07,32300000.0,0.120030,31.660535
2,2003-01-03,16.945643,954.471306,3.015529,308.536731,294.436472,0.151539,-0.000128,-0.000058,0.019037,0.0,9.142362e+06,33300000.0,0.126387,30.085849
3,2003-01-04,17.761199,954.172469,1.865083,299.395258,293.495316,0.151312,-0.000044,-0.000026,0.079633,0.0,4.418173e+06,33600000.0,0.155643,30.569864
4,2003-01-05,14.281283,954.395644,2.287668,265.010524,296.410658,0.151222,-0.000545,-0.000079,0.194655,0.0,1.449708e+07,31500000.0,0.155939,31.878895
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6553,2020-12-25,13.330685,951.670802,2.015309,292.172202,293.417897,0.132338,-0.001512,-0.000082,0.249628,0.0,1.558327e+07,28400000.0,0.144402,30.396700
6554,2020-12-26,14.027632,952.521458,1.736367,284.297801,293.992337,0.132305,-0.001512,-0.000066,0.249628,0.0,1.571960e+07,28400000.0,0.143072,32.262344
6555,2020-12-27,13.223768,951.342613,1.754041,288.307250,294.178115,0.132271,-0.001620,-0.000077,0.249628,0.0,1.594387e+07,28100000.0,0.141706,33.796627
6556,2020-12-28,12.504071,951.101661,2.012608,299.893708,294.514357,0.132201,-0.001720,-0.000089,0.249628,0.0,1.591720e+07,28400000.0,0.140171,31.748070


# Splitting the Input and Predictor Variables

In [16]:
features = ['DPT', 'AP', 'WS', 'WSD', 'AT', 'Albedo', 'evaporation_from_bare_soil_sum', 'evaporation_from_vegetation_transpiration_sum', 'NDVI', 'precipitation', 'surface_net_solar_radiation_sum', 'surface_thermal_radiation_downwards_sum', 'volumetric_soil_water_layer_1']
target = 'LST'

In [17]:
blr_X = blr_df[features].values
blr_y = blr_df[target].values

In [18]:
hyd_X = hyd_df[features].values
hyd_y = hyd_df[target].values

In [19]:
pune_X = pune_df[features].values
pune_y = pune_df[target].values

# Scaling the data

In [20]:
scaler = StandardScaler()

In [21]:
blr_X_scaled = scaler.fit_transform(blr_X)
hyd_X_scaled = scaler.transform(hyd_X)
pune_X_scaled = scaler.transform(pune_X)

# Splitting it into Training data and Testing Data

In [22]:
train_size = int(0.8 * len(blr_X_scaled))
blr_X_train = blr_X_scaled[:train_size]
blr_y_train = blr_y[:train_size]
blr_X_test = blr_X_scaled[train_size:]
blr_y_test = blr_y[train_size:]

In [23]:
train_size = int(0.8 * len(hyd_X_scaled))
hyd_X_train = hyd_X_scaled[:train_size]
hyd_y_train = hyd_y[:train_size]
hyd_X_test = hyd_X_scaled[train_size:]
hyd_y_test = hyd_y[train_size:]

In [24]:
train_size = int(0.8 * len(pune_X_scaled))
pune_X_train = pune_X_scaled[:train_size]
pune_y_train = pune_y[:train_size]
pune_X_test = pune_X_scaled[train_size:]
pune_y_test = pune_y[train_size:]

# Converting it into Pytorch Tensor for Further Analysis

In [25]:
blr_X_train_tensor  = torch.tensor(blr_X_train, dtype=torch.float32).to(device)
blr_y_train_tensor  = torch.tensor(blr_y_train, dtype=torch.float32).unsqueeze(1).to(device)
blr_X_test_tensor   = torch.tensor(blr_X_test, dtype=torch.float32).to(device)
blr_y_test_tensor   = torch.tensor(blr_y_test, dtype=torch.float32).unsqueeze(1).to(device)

hyd_X_train_tensor  = torch.tensor(hyd_X_train, dtype=torch.float32).to(device)
hyd_y_train_tensor  = torch.tensor(hyd_y_train, dtype=torch.float32).unsqueeze(1).to(device)
hyd_X_test_tensor   = torch.tensor(hyd_X_test, dtype=torch.float32).to(device)
hyd_y_test_tensor   = torch.tensor(hyd_y_test, dtype=torch.float32).unsqueeze(1).to(device)

pune_X_train_tensor = torch.tensor(pune_X_train, dtype=torch.float32).to(device)
pune_y_train_tensor = torch.tensor(pune_y_train, dtype=torch.float32).unsqueeze(1).to(device)
pune_X_test_tensor  = torch.tensor(pune_X_test, dtype=torch.float32).to(device)
pune_y_test_tensor  = torch.tensor(pune_y_test, dtype=torch.float32).unsqueeze(1).to(device)

# Defining the ANN Model

In [26]:
class ANN(nn.Module):
    def __init__(self, input_size, hidden_sizes, dropout_rate):
        super(ANN, self).__init__()
        layers = []
        in_features = input_size

        for hidden_size in hidden_sizes:
            layers.append(nn.Linear(in_features, hidden_size))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rate))
            in_features = hidden_size

        layers.append(nn.Linear(in_features, 1))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)

# Training the Model with the Hyperparameter Tuning done using Optuna

In [27]:
from tqdm import tqdm

def objective(trial, X_train_tensor, y_train_tensor, X_test_tensor, y_test_tensor):
    hidden_layer_count = trial.suggest_int("n_layers", 1, 4)
    hidden_sizes = [trial.suggest_int(f"n_units_l{i}", 50, 500) for i in range(hidden_layer_count)]
    dropout = trial.suggest_float("dropout", 0.0, 0.4) if hidden_layer_count > 1 else 0.0
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    batch_size = trial.suggest_categorical("batch_size", [8])
    epochs = trial.suggest_int("epochs", 50, 150)

    model = ANN(input_size=X_train.shape[1], hidden_sizes=hidden_sizes, dropout_rate=dropout).to(device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    dataset = torch.utils.data.TensorDataset(X_train_tensor, y_train_tensor)
    loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)

    model.train()
    for epoch in range(epochs):
        for batch_X, batch_y in loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()

    # Evaluate on test data
    model.eval()
    with torch.no_grad():
        predictions = model(X_test_tensor)
        mse = criterion(predictions, y_test_tensor).item()
    return mse

In [28]:
cities = {
    'Bangalore': (blr_X_train_tensor, blr_y_train_tensor, blr_X_test_tensor, blr_y_test_tensor),
    'Hyderabad': (hyd_X_train_tensor, hyd_y_train_tensor, hyd_X_test_tensor, hyd_y_test_tensor),
    'Pune': (pune_X_train_tensor, pune_y_train_tensor, pune_X_test_tensor, pune_y_test_tensor),
}

In [29]:
results = {}

In [30]:
for city, (X_train, y_train, X_test, y_test) in cities.items():
    print(f"Running Optuna for {city}...")
    study = optuna.create_study(direction="minimize")
    study.optimize(lambda trial: objective(trial, X_train, y_train, X_test, y_test), n_trials=30)
    
    best_trial = study.best_trial
    print(f"{city} Best MSE: {best_trial.value:.4f}")
    print(f"Best Parameters: {best_trial.params}\n")

    results[city] = best_trial

[I 2025-04-29 19:57:14,711] A new study created in memory with name: no-name-79dce8d1-c94f-4a02-8f41-3bab9d25ef8e


Running Optuna for Bangalore...


[I 2025-04-29 20:02:51,964] Trial 0 finished with value: 7.545886039733887 and parameters: {'n_layers': 4, 'n_units_l0': 150, 'n_units_l1': 166, 'n_units_l2': 342, 'n_units_l3': 346, 'dropout': 0.22270291156723082, 'lr': 0.0004746157441784593, 'batch_size': 8, 'epochs': 148}. Best is trial 0 with value: 7.545886039733887.
[I 2025-04-29 20:05:08,576] Trial 1 finished with value: 9.15914535522461 and parameters: {'n_layers': 3, 'n_units_l0': 449, 'n_units_l1': 71, 'n_units_l2': 140, 'dropout': 0.39613883990832033, 'lr': 0.0030834843510325325, 'batch_size': 8, 'epochs': 77}. Best is trial 0 with value: 7.545886039733887.
[I 2025-04-29 20:08:18,034] Trial 2 finished with value: 7.154452323913574 and parameters: {'n_layers': 2, 'n_units_l0': 394, 'n_units_l1': 475, 'dropout': 0.22676281744543242, 'lr': 0.00024260036010265802, 'batch_size': 8, 'epochs': 136}. Best is trial 2 with value: 7.154452323913574.
[I 2025-04-29 20:10:51,033] Trial 3 finished with value: 8.040435791015625 and paramete

KeyboardInterrupt: 

In [ ]:
results

# Evaluating the Model and Visualizing the Results

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [ ]:
def evaluate_predictions(y_true, y_pred):
    y_true = y_true.squeeze()
    y_pred = y_pred.squeeze()
    
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    
    # NSE
    nse = 1 - np.sum((y_true - y_pred)**2) / np.sum((y_true - np.mean(y_true))**2)
    
    # RSR = RMSE / STDEV of observed
    rsr = rmse / np.std(y_true)
    
    # PBIAS
    pbias = 100 * np.sum(y_true - y_pred) / np.sum(y_true)

    return {
        "MSE": mse,
        "MAE": mae,
        "R²": r2,
        "NSE": nse,
        "RSR": rsr,
        "PBIAS": pbias
    }

In [ ]:
def plot_predictions(y_true, y_pred, title="Prediction vs Ground Truth"):
    plt.figure(figsize=(10, 5))
    plt.plot(y_true.squeeze(), label="True", alpha=0.7)
    plt.plot(y_pred.squeeze(), label="Predicted", alpha=0.7)
    plt.title(title)
    plt.xlabel("Time Step")
    plt.ylabel("LST")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [ ]:
from optuna.visualization.matplotlib import plot_optimization_history

def plot_optuna_loss(study, title="Optuna Loss Over Trials"):
    fig = plot_optimization_history(study)
    fig.gca().set_title(title)
    plt.tight_layout()
    plt.show()

In [ ]:
def train_final_model(X_train, y_train, X_test, y_test, best_params):
    hidden_sizes = [best_params[f"n_units_l{i}"] for i in range(best_params["n_layers"])]
    model = ANN(X_train.shape[1], hidden_sizes, best_params.get("dropout", 0.0))
    optimizer = torch.optim.Adam(model.parameters(), lr=best_params["lr"])
    criterion = nn.MSELoss()

    dataset = torch.utils.data.TensorDataset(X_train, y_train).to(device)
    loader = torch.utils.data.DataLoader(dataset, batch_size=best_params["batch_size"], shuffle=True)

    model.train()
    for epoch in tqdm(range(best_params["epochs"]), desc="Training Progress", leave=True):
        with tqdm(loader, desc=f"Epoch {epoch+1}/{best_params['epochs']}", leave=False) as t:
            for X_batch, y_batch in t:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                optimizer.zero_grad()
                loss = criterion(model(X_batch), y_batch)
                loss.backward()
                optimizer.step()
                t.set_postfix(loss=loss.item())

    return model

In [ ]:
import plotly.graph_objects as go
def plot_predictions_plotly(y_true, y_pred, title="Prediction vs Ground Truth (Test Data)"):
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        y=y_true.squeeze(),
        mode='lines',
        name='True',
        line=dict(color='blue')
    ))

    fig.add_trace(go.Scatter(
        y=y_pred.squeeze(),
        mode='lines',
        name='Predicted',
        line=dict(color='orange')
    ))

    fig.update_layout(
        title=title,
        xaxis_title="Time Step (Test Set)",
        yaxis_title="LST",
        legend=dict(x=0.01, y=0.99),
        template='plotly_white',
        height=500,
        width=1000
    )

    fig.show()

In [ ]:
blr_best_params = results["Bangalore"].params
pune_best_params = results["Pune"].params
hyd_best_params = results["Hyderabad"].params

In [ ]:
blr_model = train_final_model(
    blr_X_train_tensor, blr_y_train_tensor,
    blr_X_test_tensor, blr_y_test_tensor,
    blr_best_params
)

In [ ]:
blr_model.eval()

In [ ]:
with torch.no_grad():
    y_pred_blr = blr_model(blr_X_test_tensor).cpu().numpy()
    y_true_blr = blr_y_test_tensor.cpu().numpy()

In [ ]:
metrics_blr = evaluate_predictions(y_true_blr, y_pred_blr)
print("Bangalore Metrics:")
for k, v in metrics_blr.items():
    print(f"{k}: {v:.4f}")

In [ ]:
plot_predictions_plotly(y_true_blr, y_pred_blr, title="Bangalore's Prediction vs Ground Truth (Test)")

In [ ]:
hyd_model = train_final_model(
    hyd_X_train_tensor, hyd_y_train_tensor,
    hyd_X_test_tensor, hyd_y_test_tensor,
    hyd_best_params
)

In [ ]:
hyd_model.eval()

In [ ]:
with torch.no_grad():
    y_pred_hyd = hyd_model(hyd_X_test_tensor).cpu().numpy()
    y_true_hyd = hyd_y_test_tensor.cpu().numpy()

In [ ]:
metrics_hyd = evaluate_predictions(y_true_hyd, y_pred_hyd)
print("Hyderabad Metrics:")
for k, v in metrics_hyd.items():
    print(f"{k}: {v:.4f}")

In [ ]:
plot_predictions_plotly(y_true_hyd, y_pred_hyd, title="Hyderabad's Prediction vs Ground Truth (Test)")

In [ ]:
pune_model = train_final_model(
    pune_X_train_tensor, pune_y_train_tensor,
    pune_X_test_tensor, pune_y_test_tensor,
    pune_best_params
)

In [ ]:
pune_model.eval()

In [ ]:
with torch.no_grad():
    y_pred_pune = pune_model(pune_X_test_tensor).cpu().numpy()
    y_true_pune = pune_y_test_tensor.cpu().numpy()

In [ ]:
metrics_pune = evaluate_predictions(y_true_pune, y_pred_pune)
print("Pune's Metrics:")
for k, v in metrics_pune.items():
    print(f"{k}: {v:.4f}")

In [ ]:
plot_predictions_plotly(y_true_pune, y_pred_pune, title="Pune's Prediction vs Ground Truth (Test)")

# Storing the model for future use

In [ ]:
os.makedirs('../Models/ANN', exist_ok=True)

In [ ]:
torch.save(blr_model.state_dict(), "../Models/ANN/blr_ann2_model.pth")

In [ ]:
torch.save(hyd_model.state_dict(), "../Models/ANN/hyd_ann2_model.pth")

In [ ]:
torch.save(pune_model.state_dict(), "../Models/ANN/pune_ann2_model.pth")